# whitenoise.exoplanet — TESS Transit SWNA Demo

Walks through the full exoplanet pipeline: estimate transit duration (T14),
find an empirical transit midpoint, extract a gap-filled "3T" transit window,
and run SWNA (`wn.analyze()`) on it — first for a single transit, then in
batch over a folder of pipeline CSVs.

If a real TESS pipeline CSV folder is not found, this notebook falls back to
a synthetic full-sector light curve with an injected transit dip, so it runs
end-to-end with no network access required.

In [1]:
import sys, os

# Fix Unicode output on Windows terminals
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Development path — remove this block if whitenoise is installed via pip
# (repo layout: <repo_root>/whitenoise/exoplanet/this_notebook.ipynb,
#  package import root is <repo_root>, since <repo_root>/whitenoise/__init__.py is the package)
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
_REPO_ROOT = os.path.abspath(os.path.join(_NOTEBOOK_DIR, '..', '..'))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# Remove the next line if running in an interactive Jupyter environment
matplotlib.use('Agg')

import whitenoise as wn

print('whitenoise version:', wn.__version__)
print('exoplanet functions:', [f for f in dir(wn.exoplanet) if not f.startswith('_')])

whitenoise version: 0.1.0
exoplanet functions: ['analyze_transit', 'batch_analyze_transits', 'clean_lc', 'download_transit_lc', 'estimate_T14', 'extract_transit_window', 'fill_gaps_nn', 'find_empirical_midpoint', 'io', 'list_transit_csvs', 'pipeline', 'preprocess', 'read_pipeline_csv', 'window_has_gap', 'write_pipeline_csv']


## 1. Get a light curve

`wn.exoplanet.download_transit_lc()` downloads and cleans a real TESS SPOC
light curve (requires `lightkurve` — `pip install whitenoise-swna[exoplanet]`).
It's demonstrated in its own cell below, kept separate from the rest of this
notebook so the walkthrough below runs deterministically without depending
on network access or real transit ephemeris: it uses a synthetic light curve
with an injected transit dip.

In [2]:
TARGET       = 'WASP-078'
P_DAYS       = 2.175    # orbital period (days)
RS_SUN       = 2.350    # stellar radius (solar radii)
RP_JUP       = 2.060    # planet radius (Jupiter radii)
A_AU         = 0.03670  # semi-major axis (AU)

T14_d = wn.exoplanet.estimate_T14(P_days=P_DAYS, Rs=RS_SUN, Rp=RP_JUP, a_AU=A_AU)
print(f'T14 = {T14_d:.5f} days = {T14_d*24:.2f} hours')

# Synthetic full-sector light curve with injected transit dips, so this demo
# runs deterministically without network access. See the optional
# "Downloading a real light curve" cell below for the real-data equivalent.
rng = np.random.default_rng(42)
cadence_d = 2.0 / 1440.0   # TESS 2-min cadence
time = np.arange(0.0, 6 * P_DAYS, cadence_d)
flux = 1.0 + rng.normal(0, 0.0008, size=len(time))

depth_frac = 0.0081  # 0.811% depth
N_INJECTED = 6
for n in range(N_INJECTED):
    t_mid_n = 0.5 * P_DAYS + n * P_DAYS
    in_transit = np.abs(time - t_mid_n) < T14_d / 2
    flux[in_transit] -= depth_frac

print(f'Synthetic light curve: {len(time)} points, {N_INJECTED} injected transits')

T14 = 0.22888 days = 5.49 hours
Synthetic light curve: 9396 points, 6 injected transits


## 2. Find an empirical transit midpoint

`find_empirical_midpoint()` locates a transit within a search window
without needing a forward model — a sliding scan, parabolic refinement,
and flux-weighted centroid are blended based on SNR.

In [3]:
# Search window around the first expected transit
t_lo = 0.5 * P_DAYS - P_DAYS / 4
t_hi = 0.5 * P_DAYS + P_DAYS / 4

t_mid, depth, snr, method = wn.exoplanet.find_empirical_midpoint(time, flux, t_lo, t_hi, T14_d)
print(f't_mid  = {t_mid:.6f} days')
print(f'depth  = {depth:.5f}')
print(f'SNR    = {snr:.2f}')
print(f'method = {method}')

t_mid  = 1.101670 days
depth  = 0.00794
SNR    = 5.47
method = SCAN+BLEND(SNR=5.5)


## 3. Extract a gap-filled "3T" transit window

`extract_transit_window()` cuts out `t_mid ± 1.5*T14`, fills small gaps by
nearest-neighbour, and rejects the window if too few points remain or
coverage is too low.

In [4]:
window = wn.exoplanet.extract_transit_window(time, flux, t_mid, T14_d)

if window is None:
    print('Window rejected by quality gates (out of bounds, too few points, or low coverage).')
else:
    print(f"{len(window['time'])} points  |  {window['n_filled']} NN-filled  |  "
          f"coverage={window['coverage']:.2f}")

    fig, ax = plt.subplots(figsize=(9, 3.5))
    fld = window['is_filled']
    x = (window['time'] - t_mid) * 1440.0
    ax.plot(x[~fld], window['flux'][~fld], '.', ms=3, color='#2C3E50', label='Observed')
    if fld.any():
        ax.plot(x[fld], window['flux'][fld], 'x', ms=4, color='#E74C3C', label='NN-filled')
    ax.set_xlabel('Time from midpoint (min)')
    ax.set_ylabel('Normalized flux')
    ax.set_title(f'{TARGET} — extracted transit window')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    plt.show()

494 points  |  0 NN-filled  |  coverage=1.00


C:\Users\pnayg\AppData\Local\Temp\ipykernel_57436\621296210.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Single-transit SWNA analysis

`analyze_transit()` runs the full chain — window extraction, CSV export,
`wn.analyze()`, and a diagnostic plot — in one call.

In [5]:
OUT_DIR = os.path.join(_NOTEBOOK_DIR, 'swna_results_demo')

out = wn.exoplanet.analyze_transit(
    time, flux, t_mid, T14_d,
    model='exponential',
    detrend_method=None,
    csv_path=os.path.join(OUT_DIR, f'{TARGET}_T01.csv'),
    output_dir=OUT_DIR,
    dataset_name=f'{TARGET}_T01',
)

if out['result'] and out['result'].fit:
    out['result'].summary()
else:
    print('Fitting failed — check the model choice and data quality.')


  WASP-078_T01
  --------------------------------------------------------
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\WASP-078_T01.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.21, mean=0.60).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (494 points, max_lag=247)...
✓ Fitting exponential model...


✓ Good fit  | R²(pure)=-49031.9068  R²(N·MSD)=0.9083  → N·MSD selected
✓ Done.  R² = 0.9083  |  non-Markovian, short memory


  [OK]  mu=0.8323  R2=0.9083  [non-Markovian, short memory]  (0 pts NN-filled)
══════════════════════════════════════════
 SWNA Analysis Summary
══════════════════════════════════════════
 Dataset   : WASP-078_T01
 Model     : exponential
 Points    : 494
 Lags used : 247
──────────────────────────────────────────
 Parameters:
   μ      = 0.8323  ±  0.0976
   β      = 98.6915  ±  11.7038
   N      = 0.0091  ±  0.0013
 R² (pure MSD)  = -49031.9068
 R² (N·MSD)     = 0.9083 ← selected
 (μ−1)/2   = -0.0839
 Memory    : non-Markovian, short memory  (Tey et al. 2024)
──────────────────────────────────────────
 Units     : x=time_min_from_mid, y=flux
══════════════════════════════════════════


## 5. Batch analysis over a pipeline folder

`batch_analyze_transits()` walks a `pipeline/<Planet>/<Planet>_T##.csv`
tree (as written by `write_pipeline_csv()`), running one or more SWNA
models on every transit and collecting a summary DataFrame.

In [6]:
PIPELINE_DIR = os.path.join(OUT_DIR, 'pipeline', TARGET)
os.makedirs(PIPELINE_DIR, exist_ok=True)

# Write a few more synthetic/extracted transit windows for the batch demo.
# Provenance (method, SNR) is tracked separately here rather than embedded
# in the CSV, since wn.analyze() requires a pure 2-column file.
provenance = []
n_written = 0
for n in range(6):
    t_mid_n = 0.5 * P_DAYS + n * P_DAYS
    lo_n = t_mid_n - P_DAYS / 4
    hi_n = t_mid_n + P_DAYS / 4
    if lo_n < time[0] or hi_n > time[-1]:
        continue
    tm, dep, s, meth = wn.exoplanet.find_empirical_midpoint(time, flux, lo_n, hi_n, T14_d)
    if tm is None:
        continue
    w = wn.exoplanet.extract_transit_window(time, flux, tm, T14_d)
    if w is None:
        continue
    n_written += 1
    transit_name = f'{TARGET}_T{n_written:02d}'
    wn.exoplanet.write_pipeline_csv(
        w['time'], w['flux'], tm,
        os.path.join(PIPELINE_DIR, f'{transit_name}.csv'),
    )
    provenance.append({'transit': transit_name, 't_mid': tm, 'method': meth, 'snr': s})

print(f'{n_written} transit CSV(s) written to {PIPELINE_DIR}')
for row in provenance:
    print(f"  {row['transit']}: t_mid={row['t_mid']:.6f}  SNR={row['snr']:.1f}  [{row['method']}]")

df = wn.exoplanet.batch_analyze_transits(
    os.path.join(OUT_DIR, 'pipeline'),
    models=('cosine', 'exponential'),
    output_dir=os.path.join(OUT_DIR, 'swna'),
)
df[['planet', 'transit', 'model', 'r_squared', 'regime']]

6 transit CSV(s) written to C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078
  WASP-078_T01: t_mid=1.101670  SNR=5.5  [SCAN+BLEND(SNR=5.5)]
  WASP-078_T02: t_mid=3.287735  SNR=4.6  [SCAN+BLEND(SNR=4.6)]
  WASP-078_T03: t_mid=5.416677  SNR=4.9  [SCAN+BLEND(SNR=4.9)]
  WASP-078_T04: t_mid=7.629254  SNR=5.3  [SCAN+BLEND(SNR=5.3)]
  WASP-078_T05: t_mid=9.765084  SNR=4.6  [SCAN+BLEND(SNR=4.6)]
  WASP-078_T06: t_mid=11.954879  SNR=6.6  [SCAN+BLEND(SNR=6.6)]
=== Model: COSINE ===

  WASP-078 (6 transit(s))
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T01.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.21, mean=0.60).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly re

    [OK]  WASP-078_T01            mu=0.7515  R2=0.8001  [subdiffusive]
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T02.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.79, mean=-0.34).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (495 points, max_lag=247)...
✓ Fitting cosine model...
✓ Good fit  | R²(pure)=-2714470931.5379  R²(N·MSD)=0.8102  → N·MSD selected
✓ Done.  R² = 0.8102  |  subdiffusive


    [OK]  WASP-078_T02            mu=0.7541  R2=0.8102  [subdiffusive]
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T03.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.79, mean=-0.02).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (495 points, max_lag=247)...
✓ Fitting cosine model...
✓ Good fit  | R²(pure)=-2604781183.0640  R²(N·MSD)=0.9870  → N·MSD selected
✓ Done.  R² = 0.9870  |  superdiffusive


    [OK]  WASP-078_T03            mu=1.3194  R2=0.9870  [superdiffusive]
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T04.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.79, mean=-0.13).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (495 points, max_lag=247)...
✓ Fitting cosine model...
✓ Good fit  | R²(pure)=-2605667807.6514  R²(N·MSD)=0.9861  → N·MSD selected
✓ Done.  R² = 0.9861  |  superdiffusive


    [OK]  WASP-078_T04            mu=1.3284  R2=0.9861  [superdiffusive]
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T05.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.79, mean=0.28).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (495 points, max_lag=247)...
✓ Fitting cosine model...
✓ Good fit  | R²(pure)=-2943697278.5585  R²(N·MSD)=0.8029  → N·MSD selected
✓ Done.  R² = 0.8029  |  subdiffusive


...

    [OK]  WASP-078_T04            mu=0.8234  R2=0.9080  [non-Markovian, short memory]
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T05.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.79, mean=0.28).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (495 points, max_lag=247)...
✓ Fitting exponential model...
✓ Good fit  | R²(pure)=-51403.8489  R²(N·MSD)=0.9032  → N·MSD selected
✓ Done.  R² = 0.9032  |  non-Markovian, sho

    [OK]  WASP-078_T05            mu=0.8038  R2=0.9032  [non-Markovian, short memory]
✓ Loading: C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\pipeline\WASP-078\WASP-078_T06.csv
⚠  Column order warning: column 1 ('time_min_from_mid') has unusually large spread (std=285.21, mean=-0.03).
   If your columns are swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
⚠  Column 1 name warning: 'time_min_from_mid' is not a commonly recognized time or index name.
   If column 1 is a non-time independent variable (e.g. distance, frequency, voltage), this warning is expected — ignore it.
   If your columns are accidentally swapped, re-save the CSV with the independent variable in column 1 and the observable in column 2.
✓ Computing MSD  (494 points, max_lag=247)...
✓ Fitting exponential model...
✓ Good fit  | R²(pure)=-45105.5987  R²(N·MSD)=0.9058  → N·MSD selected
✓ Done.  R² = 0.9058  |  non-Markovian, sh

    [OK]  WASP-078_T06            mu=0.8295  R2=0.9058  [non-Markovian, short memory]

Summary CSV  --> C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\swna\swna_summary.csv


Summary XLSX --> C:\Users\pnayg\Desktop\latest working whitenoise-dev\whitenoise\exoplanet\swna_results_demo\swna\swna_summary.xlsx


,planet,transit,model,r_squared,regime
0,WASP-078,WASP-078_T01,cosine,0.800072,subdiffusive
1,WASP-078,WASP-078_T02,cosine,0.810229,subdiffusive
2,WASP-078,WASP-078_T03,cosine,0.987021,superdiffusive
3,WASP-078,WASP-078_T04,cosine,0.986133,superdiffusive
4,WASP-078,WASP-078_T05,cosine,0.802946,subdiffusive
5,WASP-078,WASP-078_T06,cosine,0.986262,superdiffusive
6,WASP-078,WASP-078_T01,exponential,0.908305,"non-Markovian, short memory"
7,WASP-078,WASP-078_T02,exponential,0.914052,"non-Markovian, short memory"
8,WASP-078,WASP-078_T03,exponential,0.911459,"non-Markovian, short memory"
9,WASP-078,WASP-078_T04,exponential,0.907971,"non-Markovian, short memory"


## 6. (Optional) Downloading a real light curve

`download_transit_lc()` downloads every available TESS SPOC sector for a
target and picks the one with the most estimated transits. Requires
`lightkurve` and network access — this cell is illustrative only and does
not feed into the rest of the notebook, since real transit timing needs a
proper ephemeris (period + reference epoch) rather than the `t_mid = 0.5*P`
guess used for the synthetic demo above.

In [7]:
try:
    real_time, real_flux, real_meta = wn.exoplanet.download_transit_lc(TARGET, period_days=P_DAYS)
    print(f'Downloaded sector {real_meta["sector"]}, {len(real_time)} points, '
          f'~{real_meta["n_transits"]:.1f} transits')
    print(f'BTJD range: [{real_time[0]:.3f}, {real_time[-1]:.3f}]')
except Exception as exc:
    print(f'Download unavailable in this environment: {exc}')

c:\Users\pnayg\Desktop\cvif-astro-p1\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


Download unavailable in this environment: HTTPSConnectionPool(host='mast.stsci.edu', port=443): Max retries exceeded with url: /portal/Mashup/Mashup.asmx/columnsconfig (Caused by NameResolutionError("HTTPSConnection(host='mast.stsci.edu', port=443): Failed to resolve 'mast.stsci.edu' ([Errno 11001] getaddrinfo failed)"))
